# Assemble Data

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.ndimage import uniform_filter1d  # simple smoothing
import re

/home/simonhans/anaconda3/envs/GrapeExpectations/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [2]:
pd.set_option('display.max_columns', None)  # show all columns
# pd.set_option('display.max_rows', None)  # show all columns
# import os
# os.chdir('..')

### Start with plot features

In [3]:
# get elevation features
df = pd.read_pickle('../data/plot_elev_features.pkl')

In [4]:
df

,plot_id,slope_mean,slope_min,slope_max,curve_mean,curve_min,curve_max,pro_curve_mean,pro_curve_min,pro_curve_max,plan_curve_mean,plan_curve_min,plan_curve_max,aspect_mean,aspect_min,aspect_max,elev_min,elev_max,elev_mean,elev_dev_min,elev_dev_max,elev_dev_mean,geometry,total_relief,area_m2,area_ha
0,0,5.020058,3.925700,6.804783,0.000489,-0.009361,0.013225,0.000620,-0.010633,0.013191,0.000489,-0.009361,0.013225,175.379315,155.373535,190.393188,207.326294,208.851974,208.022307,22.362457,23.888138,23.058465,"POLYGON ((-119.78208 45.87245, -119.78215 45.8...",1.525681,451.141838,0.045114
1,1,5.289605,3.356515,8.816703,-0.000258,-0.014769,0.013737,-0.001123,-0.016449,0.012478,-0.000258,-0.014769,0.013737,169.123825,144.559204,188.072983,206.874710,208.469513,207.776839,21.910873,23.505676,22.812984,"POLYGON ((-119.78179 45.87245, -119.78167 45.8...",1.594803,381.909845,0.038191
2,2,7.150476,4.206255,11.814886,0.000065,-0.030229,0.034444,-0.000754,-0.043223,0.027175,0.000065,-0.030229,0.034444,152.046614,135.318848,175.093628,205.507690,207.790955,206.759655,20.543854,22.827118,21.795817,"POLYGON ((-119.78139 45.87246, -119.78129 45.8...",2.283264,304.757142,0.030476
3,3,9.178217,5.255136,12.657069,0.000241,-0.022650,0.018867,0.000610,-0.020701,0.012653,0.000241,-0.022650,0.018867,175.202423,159.699402,189.218704,210.499161,214.308624,212.434447,25.535324,29.344788,27.470613,"POLYGON ((-119.78299 45.87287, -119.78300 45.8...",3.809464,416.261511,0.041626
4,4,6.641992,4.568419,10.110537,0.000753,-0.013419,0.014943,0.000611,-0.011132,0.012551,0.000753,-0.013419,0.014943,173.372281,156.050049,189.273071,209.319901,213.504730,211.261849,24.356064,28.540894,26.298020,"POLYGON ((-119.78263 45.87269, -119.78269 45.8...",4.184830,934.859055,0.093486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3593,3593,6.524146,4.537724,8.501439,-0.001274,-0.024254,0.019990,-0.001476,-0.019637,0.020838,-0.001274,-0.024254,0.019990,163.878749,140.306885,186.459793,228.098480,232.209412,230.327266,43.134644,47.245575,45.363431,"POLYGON ((-119.77610 45.87519, -119.77626 45.8...",4.110931,544.855651,0.054486
3594,3594,3.070807,1.769737,4.388859,0.000577,-0.013051,0.017839,0.000672,-0.015131,0.010445,0.000577,-0.013051,0.017839,176.192068,142.778198,228.122147,234.638351,235.519440,235.030851,49.674515,50.555603,50.067012,"POLYGON ((-119.77718 45.87568, -119.77718 45.8...",0.881088,319.126138,0.031913
3595,3595,3.290681,1.204210,6.403341,-0.001655,-0.029511,0.026497,-0.002227,-0.026850,0.022847,-0.001655,-0.029511,0.026497,210.650471,141.063812,265.084839,235.022568,236.736877,235.759766,50.058731,51.773041,50.795942,"POLYGON ((-119.77685 45.87573, -119.77680 45.8...",1.714310,591.653344,0.059165
3596,3596,6.540738,1.204210,9.352532,-0.000439,-0.029511,0.027310,-0.001295,-0.025244,0.019796,-0.000439,-0.029511,0.027310,140.301358,110.309143,226.955688,232.924850,236.760712,235.060714,47.961014,51.796875,50.096894,"POLYGON ((-119.77652 45.87579, -119.77642 45.8...",3.835861,847.012539,0.084701


### Compute vectors for aspect and slope, and some interaction terms

In [5]:
# Aspect: convert to radians and compute sin/cos
df['aspect_min_cos'] = np.cos(np.radians(df['aspect_min']))
df['aspect_min_sin'] = np.sin(np.radians(df['aspect_min']))

df['aspect_max_cos'] = np.cos(np.radians(df['aspect_max']))
df['aspect_max_sin'] = np.sin(np.radians(df['aspect_max']))

df['aspect_mean_cos'] = np.cos(np.radians(df['aspect_mean']))
df['aspect_mean_sin'] = np.sin(np.radians(df['aspect_mean']))

# Drop raw aspect values
df = df.drop(columns=['aspect_min', 'aspect_max', 'aspect_mean'])

df['slope_rad'] = np.radians(df['slope_mean'])
df['slope_grad'] = np.tan(df['slope_rad'])


df['slope_x'] = df['slope_grad'] * df['aspect_mean_cos']
df['slope_y'] = df['slope_grad'] * df['aspect_mean_sin']

df = df.drop(columns = ['slope_mean','slope_min','slope_max'])

In [6]:
df

,plot_id,curve_mean,curve_min,curve_max,pro_curve_mean,pro_curve_min,pro_curve_max,plan_curve_mean,plan_curve_min,plan_curve_max,elev_min,elev_max,elev_mean,elev_dev_min,elev_dev_max,elev_dev_mean,geometry,total_relief,area_m2,area_ha,aspect_min_cos,aspect_min_sin,aspect_max_cos,aspect_max_sin,aspect_mean_cos,aspect_mean_sin,slope_rad,slope_grad,slope_x,slope_y
0,0,0.000489,-0.009361,0.013225,0.000620,-0.010633,0.013191,0.000489,-0.009361,0.013225,207.326294,208.851974,208.022307,22.362457,23.888138,23.058465,"POLYGON ((-119.78208 45.87245, -119.78215 45.8...",1.525681,451.141838,0.045114,-0.909044,0.416701,-0.983593,-0.180402,-0.996750,0.080559,0.087617,0.087841,-0.087556,0.007076
1,1,-0.000258,-0.014769,0.013737,-0.001123,-0.016449,0.012478,-0.000258,-0.014769,0.013737,206.874710,208.469513,207.776839,21.910873,23.505676,22.812984,"POLYGON ((-119.78179 45.87245, -119.78167 45.8...",1.594803,381.909845,0.038191,-0.814715,0.579861,-0.990090,-0.140434,-0.982037,0.188687,0.092321,0.092584,-0.090921,0.017469
2,2,0.000065,-0.030229,0.034444,-0.000754,-0.043223,0.027175,0.000065,-0.030229,0.034444,205.507690,207.790955,206.759655,20.543854,22.827118,21.795817,"POLYGON ((-119.78139 45.87246, -119.78129 45.8...",2.283264,304.757142,0.030476,-0.711031,0.703161,-0.996336,0.085528,-0.883329,0.468753,0.124799,0.125451,-0.110815,0.058806
3,3,0.000241,-0.022650,0.018867,0.000610,-0.020701,0.012653,0.000241,-0.022650,0.018867,210.499161,214.308624,212.434447,25.535324,29.344788,27.470613,"POLYGON ((-119.78299 45.87287, -119.78300 45.8...",3.809464,416.261511,0.041626,-0.937885,0.346945,-0.987084,-0.160203,-0.996496,0.083636,0.160190,0.161575,-0.161008,0.013513
4,4,0.000753,-0.013419,0.014943,0.000611,-0.011132,0.012551,0.000753,-0.013419,0.014943,209.319901,213.504730,211.261849,24.356064,28.540894,26.298020,"POLYGON ((-119.78263 45.87269, -119.78269 45.8...",4.184830,934.859055,0.093486,-0.913900,0.405938,-0.986932,-0.161140,-0.993317,0.115418,0.115925,0.116447,-0.115669,0.013440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3593,3593,-0.001274,-0.024254,0.019990,-0.001476,-0.019637,0.020838,-0.001274,-0.024254,0.019990,228.098480,232.209412,230.327266,43.134644,47.245575,45.363431,"POLYGON ((-119.77610 45.87519, -119.77626 45.8...",4.110931,544.855651,0.054486,-0.769476,0.638675,-0.993651,-0.112506,-0.960676,0.277671,0.113868,0.114363,-0.109865,0.031755
3594,3594,0.000577,-0.013051,0.017839,0.000672,-0.015131,0.010445,0.000577,-0.013051,0.017839,234.638351,235.519440,235.030851,49.674515,50.555603,50.067012,"POLYGON ((-119.77718 45.87568, -119.77718 45.8...",0.881088,319.126138,0.031913,-0.796300,0.604902,-0.667545,-0.744570,-0.997792,0.066412,0.053596,0.053647,-0.053529,0.003563
3595,3595,-0.001655,-0.029511,0.026497,-0.002227,-0.026850,0.022847,-0.001655,-0.029511,0.026497,235.022568,236.736877,235.759766,50.058731,51.773041,50.795942,"POLYGON ((-119.77685 45.87573, -119.77680 45.8...",1.714310,591.653344,0.059165,-0.777846,0.628454,-0.085681,-0.996323,-0.860293,-0.509799,0.057433,0.057496,-0.049464,-0.029312
3596,3596,-0.000439,-0.029511,0.027310,-0.001295,-0.025244,0.019796,-0.000439,-0.029511,0.027310,232.924850,236.760712,235.060714,47.961014,51.796875,50.096894,"POLYGON ((-119.77652 45.87579, -119.77642 45.8...",3.835861,847.012539,0.084701,-0.347085,0.937834,-0.682564,-0.730826,-0.769415,0.638750,0.114157,0.114656,-0.088218,0.073236


## Now add NDVI for each plot to features

### open up the filtered and smoothed ndvi df

In [9]:
veg_agg = pd.read_pickle('../data_wrangling/data/ndvi/plots/final_df.pkl')
keep_cols = ['plot_id', 'year']

FileNotFoundError: [Errno 2] No such file or directory: '../data_wrangling/data/ndvi/plots/final_df.pkl'

In [ ]:
veg_agg = veg_agg.dropna(axis = 1)
veg_agg

In [ ]:
df = df.merge(veg_agg, how = 'inner', on = 'plot_id')

In [ ]:
df

### Now let's add some weather information

First load the wather that has been unzipped and clipped to the vineyard

In [10]:
weather = pd.read_pickle('../data/PRISM/df.pkl')

weather = (
    weather
    .groupby("date", as_index=False)
    .first()
)

weather['date'] = pd.to_datetime(weather['date'])
weather['doy'] = weather['date'].dt.dayofyear
weather['year'] = weather['date'].dt.year

weather


,date,ppt,tmax,tmean,tmin,vpdmax,vpdmin,doy,year
0,2016-01-01,0.000000,-4.070362,-5.326095,-6.582299,1.192378,0.604992,1,2016
1,2016-01-02,0.000000,-4.908142,-6.383614,-7.859292,1.179189,0.600425,2,2016
2,2016-01-03,1.016748,-4.797142,-6.399859,-8.004236,0.875575,0.300236,3,2016
3,2016-01-04,0.000000,-2.663181,-4.297945,-5.934236,1.153016,0.302661,4,2016
4,2016-01-05,0.433661,-0.743622,-2.255756,-3.768228,0.723819,0.093764,5,2016
...,...,...,...,...,...,...,...,...,...
1821,2020-12-26,1.237079,5.121992,1.497289,-2.127346,1.915047,0.039181,361,2020
1822,2020-12-27,0.000000,2.565496,-0.461774,-3.489197,1.192134,0.120787,362,2020
1823,2020-12-28,0.000000,1.647661,-0.097731,-1.843236,0.848346,0.182457,363,2020
1824,2020-12-29,0.776843,1.894780,0.731803,-0.431173,1.670961,0.201213,364,2020


### Crack at some frost and growing degree days cumulative per month

In [ ]:
# Compute frost days and GDD
weather['frost'] = (weather['tmin'] < 0)
weather['gdd'] = (weather['tmean'] - 10).clip(lower=0)


In [ ]:
# Compute cumulative GDD for each month
weather['week'] = weather['date'].dt.week
cumulative_gdd = weather.groupby([ 'year', 'week'])['gdd'].sum().groupby(level=[0,1]).cumsum()
# Reset index to turn MultiIndex into columns
cumulative_gdd = cumulative_gdd.rename('cumulative_gdd').reset_index()

# Merge back to weather
weather = weather.merge(
    cumulative_gdd,
    on=['year', 'week'],
    how='left'
)


weather

In [ ]:
weather = weather[
    (weather['week'] > 27) &
    (weather['week'] < 44)
].copy()

In [ ]:
# Define which aggregations to use
agg_funcs = {
    'ppt': 'sum',
    'tmax': 'max',
    'tmin': 'min',
    'tmean': 'mean',
    'vpdmax': 'max',
    'vpdmin': 'min',
    'cumulative_gdd': 'max'
}

# Aggregate to long-form first (year, week, variables)
weekly_long = weather.groupby(['year','week']).agg(agg_funcs).reset_index()

# Pivot to wide form (one row per year, columns per week)
weekly_wide = pd.DataFrame({'year': weekly_long['year'].unique()})

for col in ['ppt','tmax','tmin','tmean','vpdmax','vpdmin','cumulative_gdd']:
    pivoted = weekly_long.pivot(index='year', columns='week', values=col)
    # Rename columns to include variable name
    pivoted.columns = [f'{col}_{w}' for w in pivoted.columns]
    weekly_wide = weekly_wide.merge(pivoted, on='year', how='left')

weekly_wide.reset_index(drop=True, inplace=True)

In [ ]:
weekly_wide

### Now we can combine data.

In [ ]:
df['year'] = df['year'].astype(int)
df = pd.merge(df, weekly_wide, how = 'inner', on = 'year')

In [ ]:
df

In [ ]:
# # Optional: could also add slope magnitude transforms
# df['slope_squared'] = df['slope_mean'] ** 2
# df['slope_log'] = np.log1p(df['slope_mean'])  # log(1 + slope)

for i in range(28, 44, 1):
    
    df[f'water_availability_{i}'] = df[f'ppt_{i}'] / (1 +  df[f'cumulative_gdd_{i}'])
    df[f'diurnal_temp_range_{i}'] = df[f'tmax_{i}'] / df[f'tmin_{i}']
    df[f'stress_index{i}'] = df[f'vpdmax_{i}'] / (df[f'ppt_{i}'] + 0.1)
df['local_relief'] = df['elev_mean'] - df['elev_min']

df['total_relief_log'] = np.log1p(df['total_relief'])

In [ ]:
soil = pd.read_pickle('../data/soil/plot_summary.pkl')

In [ ]:
df = pd.merge(df, soil, how = 'inner', on = 'plot_id')

In [ ]:
df = pd.DataFrame(df.drop(columns = 'geometry'))

In [ ]:
df

In [ ]:
df.to_pickle('../data/df.pkl')